# SIREN with OnDeviceSignalLoader

This notebook demonstrates fitting a SIREN INR to an image using the new `OnDeviceSignalLoader` and `make_batched_signal_loader` utilities. Instead of the standard `DataLoader(BatchedNDSignalLoader(...))` pipeline that transfers batches from CPU→GPU every iteration, `OnDeviceSignalLoader` pins the full signal on the target device once and samples batches entirely on-device.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import numpy as np
import skimage.data, skimage.transform
from matplotlib import pyplot as plt

In [ ]:
import alpine
from alpine.models import Siren
from alpine.dataloaders import (
    BatchedNDSignalLoader,
    OnDeviceSignalLoader,
    make_batched_signal_loader,
)

## 1. Load and prepare the image signal

In [ ]:
H, W = 256, 256
gt_img = skimage.transform.resize(skimage.data.astronaut(), (H, W)).astype(np.float32)
print(f"Image shape: {gt_img.shape}  (H x W x C)")

plt.figure(figsize=(4, 4))
plt.imshow(gt_img)
plt.axis("off")
plt.title("Ground truth")
plt.show()

## 2. Create an OnDeviceSignalLoader directly

`OnDeviceSignalLoader` moves coordinates and signal to the target device once. Iteration uses `torch.randperm` on-device — no CPU→GPU transfers per batch.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
grid_dims = (H, W)
batch_size = 1024
epochs = 500

loader = OnDeviceSignalLoader(
    signal=gt_img,
    grid_dims=grid_dims,
    batch_size=batch_size,
    device=device,
    bounds=(-1, 1),
    normalize_signal=True,
)

print(f"Device   : {device}")
print(f"Batches  : {len(loader)}")
print(f"Coords   : {loader.coords.shape}  on {loader.coords.device}")
print(f"Signal   : {loader.signal.shape}  on {loader.signal.device}")
print(f"Memory   : {OnDeviceSignalLoader.estimate_memory_bytes(gt_img, grid_dims) / 1024:.1f} KB")

## 3. Build and compile the SIREN model

In [ ]:
model = Siren(
    in_features=2,
    out_features=3,
    hidden_features=256,
    hidden_layers=5,
    outermost_linear=True,
).to(device)

model.compile()
print(model)

## 4. Train with `fit_signal(dataloader=...)` using our OnDeviceSignalLoader

`OnDeviceSignalLoader` is directly iterable just like a `DataLoader`, so it plugs straight into `fit_signal(dataloader=...)`. The `.to(device)` calls inside `_fit_signal_dataloader` are no-ops since the tensors are already on-device.

In [ ]:
outputs = model.fit_signal(
    dataloader=loader,
    n_iters=epochs,
    enable_tqdm=True,
    track_loss_history=True,
)

print(f"\nFinal loss: {outputs['loss']:.6f}")

## 5. Visualize loss curve and reconstructed image

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Loss curve
axes[0].plot(outputs["loss_history"])
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training loss")

# Ground truth
axes[1].imshow(gt_img)
axes[1].axis("off")
axes[1].set_title("Ground truth")

# Reconstruction via render
from alpine.models.utils import get_coords_spatial

coords_full = get_coords_spatial(H, W).to(device)[None, ...]
with torch.no_grad():
    rendered = model(coords_full)["output"]

recon = rendered.cpu().numpy().reshape(H, W, 3).clip(0, 1)
axes[2].imshow(recon)
axes[2].axis("off")
axes[2].set_title("Reconstruction")

plt.tight_layout()
plt.show()

## 6. Use `make_batched_signal_loader` factory

The factory automatically picks the best loader for the device. On CUDA it checks available memory and returns an `OnDeviceSignalLoader`; on CPU (or with `force_cpu=True`) it returns a standard `DataLoader(BatchedNDSignalLoader(...))`.

In [ ]:
loader_auto = make_batched_signal_loader(
    signal=gt_img,
    grid_dims=grid_dims,
    batch_size=batch_size,
    device=device,
    bounds=(-1, 1),
    normalize_signal=True,
)
print(f"Factory returned: {type(loader_auto).__name__}")

# Force CPU path for comparison
loader_cpu = make_batched_signal_loader(
    signal=gt_img,
    grid_dims=grid_dims,
    batch_size=batch_size,
    device="cpu",
    force_cpu=True,
)
print(f"CPU fallback   : {type(loader_cpu).__name__}")

## 7. Timing comparison: OnDeviceSignalLoader vs DataLoader

Train identical models from the same initial weights with both loaders and compare wall-clock time.

In [ ]:
import copy, time

timing_epochs = 200
timing_batch = 1024

# Identical starting weights
model_a = Siren(in_features=2, out_features=3, hidden_features=256, hidden_layers=5, outermost_linear=True).to(device)
model_b = copy.deepcopy(model_a)

model_a.compile()
model_b.compile()

# On-device loader
loader_ondev = OnDeviceSignalLoader(gt_img, grid_dims, timing_batch, device)

# Standard DataLoader
dataset_cpu = BatchedNDSignalLoader(gt_img, grid_dims)
loader_dl = torch.utils.data.DataLoader(dataset_cpu, batch_size=timing_batch, shuffle=True, pin_memory=False)

# --- Benchmark OnDeviceSignalLoader ---
if device.type == "cuda":
    torch.cuda.synchronize()
t0 = time.perf_counter()
out_a = model_a.fit_signal(dataloader=loader_ondev, n_iters=timing_epochs, enable_tqdm=False)
if device.type == "cuda":
    torch.cuda.synchronize()
time_ondev = time.perf_counter() - t0

# --- Benchmark DataLoader ---
if device.type == "cuda":
    torch.cuda.synchronize()
t0 = time.perf_counter()
out_b = model_b.fit_signal(dataloader=loader_dl, n_iters=timing_epochs, enable_tqdm=False)
if device.type == "cuda":
    torch.cuda.synchronize()
time_dl = time.perf_counter() - t0

print(f"{'Metric':<25} {'OnDevice':>12} {'DataLoader':>12}")
print("-" * 51)
print(f"{'Total time (s)':<25} {time_ondev:>12.4f} {time_dl:>12.4f}")
print(f"{'Avg iter (ms)':<25} {time_ondev/timing_epochs*1000:>12.4f} {time_dl/timing_epochs*1000:>12.4f}")
print(f"{'Final loss':<25} {out_a['loss']:>12.6f} {out_b['loss']:>12.6f}")